# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata is a Dataset object, not a dictionary.
metadata = dataset.metadata
print(f"{metadata.name}")
print(f"Description: {metadata.description}")

# Optional: Pretty-print summary, keywords, license, etc.
print("\nSummary:")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

The FAIR<sup>2</sup> dataset comprises several tables ("record sets"):
- Table 1: Clinical features (age at diagnosis, height, clitomegaly, hirsutism, alopecia, acne, oligomenorrhea, infertility)
- Table 2: Hormone levels, adrenal CT, and gynecological ultrasound
- Table 3: Genetic variants (chromosomal location, variant details, frequency, zygosity, pathogenicity, inferred phenotype)

We'll discover the available record sets and their fields using `mlcroissant`.

In [ ]:
# Print available record sets and fields (referenced by @id)
record_sets = dataset.record_sets

print('Available Record Sets:')
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print(f"  Columns:")
    for col in rs.columns:
        print(f"    - Column name: {col.name}, @id: {col.id}, sourceField: {col.source_field}, columnType: {col.column_type}")
    print()

# For demonstration, print a few records from each record set
for rs in record_sets:
    print(f"\nRecords for RecordSet '{rs.name}' (@id: {rs.id}):")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        # Show only the first 2 records per record set for brevity
        if i >= 2:
            break
        pprint.pprint(record)


## 3. Data Extraction
Load data from each record set into DataFrames for analysis. Only `@id` is used to reference record sets and fields.

In [ ]:
# Extract data from all record sets to DataFrames
dataframes = {}

for rs in dataset.record_sets:
    # Use @id from the record set
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded DataFrame for RecordSet '{rs.name}' (@id: {rs.id}) with columns:")
    print(df.columns.tolist())
    print(df.head(2), "\n")

# Choose the main clinical features RecordSet for further EDA
# Example placeholder: Assume the first record set is clinical features table
primary_rs_id = dataset.record_sets[0].id
primary_df = dataframes[primary_rs_id]
print(f"\nPrimary DataFrame columns for RecordSet [@id]: {primary_rs_id}")
print(primary_df.columns.tolist())
primary_df.head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate EDA using fields referenced via their `@id`. This includes filtering, normalization, and grouping.

Let's select a numeric field (e.g., age at diagnosis or height), filter for records above a threshold, normalize the field, and optionally group by a categorical attribute.

In [ ]:
# Identify relevant numeric and group fields by @id
clinical_rs = dataset.record_sets[0]
numeric_field_id = None
group_field_id = None

# For demonstration, search for 'age' and 'hirsutism' field @ids
for field in clinical_rs.fields:
    if 'age' in field.name.lower():
        numeric_field_id = field.id
    if 'hirsutism' in field.name.lower():
        group_field_id = field.id

print(f"Numeric field (for filter/normalize): {numeric_field_id}\nGroup field (for grouping): {group_field_id}")

# EDA: Filtering
if numeric_field_id in primary_df.columns:
    threshold = 20
    filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (mean):")
        print(grouped_df.head())
else:
    print("Numeric field not found in primary DataFrame.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Example: Histogram of age at diagnosis, boxplot of height, scatterplot between numeric fields, etc. Field references use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization by field @id
if numeric_field_id and numeric_field_id in primary_df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(primary_df[numeric_field_id], bins=5, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (e.g., age at diagnosis)")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Frequency")
    plt.show()
    
    # If another numeric field exists, show scatterplot
    height_field_id = None
    for field in clinical_rs.fields:
        if 'height' in field.name.lower():
            height_field_id = field.id
    if height_field_id and height_field_id in primary_df.columns:
        plt.figure(figsize=(6,4))
        sns.scatterplot(x=primary_df[height_field_id], y=primary_df[numeric_field_id])
        plt.xlabel(f"{height_field_id}")
        plt.ylabel(f"{numeric_field_id}")
        plt.title(f"{numeric_field_id} vs {height_field_id}")
        plt.show()


## 6. Conclusion
This notebook demonstrates loading and exploring the FAIR<sup>2</sup> genotype–phenotype dataset using `mlcroissant`. All record sets, fields, and columns are referenced using their `@id`, ensuring consistency and reproducibility.
Key findings:
- The dataset includes clinical data for three female patients with NC-21OHD and infertility.
- Data includes structured record sets for clinical features, hormone levels, imaging, and genetic variants.
- Example EDA and visualization are performed using numeric and categorical field `@id`s.
Due to the small sample size and sensitive information, caution is recommended when generalizing or building AI models.